# MoeSekai Story Studio Bilibili PV — 花里实乃理 (`mnr`) Drama 3 一键配音工作流

本 Notebook 完整复用 `moesekai-tts` 已在 **Google Colab A100（Python 3.13 宿主 + CUDA 13.0）** 上实机跑通的 **Fish Audio Drama 3 INT4 (`fishaudio/drama-3-int4`) + `SGLang-Omni`** 推理管线，一键生成 MoeSekai Story Studio B 站宣传片全部 8 幕 / 16 句花里みのり (`pjsk_05_minori`) 无损 WAV 配音包。

### 运行前必读
1. **GPU 硬件硬性要求**：请点击顶部菜单 **代码执行程序 → 更改运行时类型 → 选择 `A100 GPU`（A100-40GB 或 A100-80GB）**。
   - `fishaudio/drama-3-int4`（22.35 GiB）需要 **`>= 32 GiB` 显存**；**L4 (24GB) / T4 (16GB) 必 OOM**，Step 1 会自动检测并立即中止以防空烧 CCU。
2. **参考干声 (`pjsk_05_minori/v1/ref.wav`) 加载方式**（Step 4 支持 3 选 1，**严禁硬编码 Token**）：
   - `upload`（推荐）：直接上传本机 `<moesekai-tts>/.local_voices_out/voices/pjsk_05_minori/v1/ref.wav`
   - `drive`：挂载 Google Drive 读取已上传的 `ref.wav`
   - `hf`：运行时交互输入（或通过 Colab Secrets `HF_TOKEN`）从私有数据集 `exmeaning/moesekai-pjsk-tts-voices` 拉取
3. **关于冷启动耗时与 `sm_80` JIT 缓存加速**：
   - 环境安装 + 模型下载 + 启动服务约 5 分钟；
   - Step 2 内置支持直接从私有 HF Dataset `exmeaning/moesekai-pjsk-tts-voices` 的 `jit-cache/sglang-0.5.23-cu130-sm80.tar.gz`（16.34 MiB，80 SGLang + 546 Triton 核）**1 秒恢复 `sm_80` 预编译 JIT 缓存**（也可从 Google Drive 恢复或留空冷编译），启用后全流程压缩至 **~5.5 分钟**！未启用缓存时第 1 句一次性冷编译约 10–18 分钟，随后全部 16 句 × 多抽卡版本在 **1–2 分钟内**批量完成！
4. **交付产物**：
   - 下载 `/content/mnr_voiceover_pack.zip`（内含 **48 kHz 单声道 16-bit PCM 无损 WAV** `-16 LUFS` 归一化音频、`timings.json`、`script.json`、`pack_manifest.json`）
   - 回到本机项目根目录运行：
     ```bash
     python promo/voiceover/unpack_colab_voice.py --zip path/to/mnr_voiceover_pack.zip --render
     ```

## Step 1 · 显存前置检查（严禁 L4/T4，仅限 A100-40GB / A100-80GB）

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

MIN_VRAM_MIB = 32 * 1024        # 32 GiB hard floor for fishaudio/drama-3-int4
HIGH_VRAM_MIB = 70 * 1024       # A100-80GB threshold
FORBIDDEN_GPU_RE = re.compile(r"(?:^|[^A-Z0-9])(L4|T4|P4|K80|P100|V100)(?:$|[^A-Z0-9])", re.IGNORECASE)


def query_gpu_info() -> dict:
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,memory.free,driver_version,compute_cap",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            timeout=15,
        ).strip()
    except Exception as exc:
        raise RuntimeError(
            "未检测到 NVIDIA GPU！请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 GPU。"
        ) from exc

    lines = [ln.strip() for ln in out.splitlines() if ln.strip()]
    if not lines:
        raise RuntimeError("nvidia-smi 返回空 GPU 列表，请切换到 A100 GPU 运行时。")
    parts = [p.strip() for p in lines[0].split(",")]
    return {
        "gpu_name": parts[0],
        "vram_total_mib": int(float(parts[1])),
        "vram_used_mib": int(float(parts[2])),
        "vram_free_mib": int(float(parts[3])),
        "driver_version": parts[4],
        "compute_cap": parts[5],
    }


GPU_INFO = query_gpu_info()
print("Detected GPU:", json.dumps(GPU_INFO, ensure_ascii=False, indent=2))

if FORBIDDEN_GPU_RE.search(GPU_INFO["gpu_name"]):
    raise RuntimeError(
        f"禁止的 GPU 型号：{GPU_INFO['gpu_name']} ({GPU_INFO['vram_total_mib']} MiB)！"
        " L4/T4 等显卡无法运行 fishaudio/drama-3-int4（必 OOM），请立即切换到 A100-40GB 或 A100-80GB！"
    )

if GPU_INFO["vram_total_mib"] < MIN_VRAM_MIB:
    raise RuntimeError(
        f"显存不足：{GPU_INFO['gpu_name']} 仅有 {GPU_INFO['vram_total_mib'] / 1024:.1f} GiB，"
        "最低要求 32 GiB（A100-40GB / A100-80GB）。已自动中止以防空烧 CCU！"
    )

if GPU_INFO["vram_total_mib"] >= HIGH_VRAM_MIB:
    GPU_PROFILE = {
        "tier": "A100-80GB",
        "tts_engine_gpu_memory_fraction": 0.62,
        "vocoder_gpu_memory_fraction": 0.20,
        "preprocessing_total_reserve_bytes": "",
        "default_concurrency": 8,
    }
else:
    GPU_PROFILE = {
        "tier": "A100-40GB",
        "tts_engine_gpu_memory_fraction": 0.55,
        "vocoder_gpu_memory_fraction": 0.15,
        "preprocessing_total_reserve_bytes": "3GiB",
        "default_concurrency": 4,
    }

print(f"GPU Gate PASSED -> tier={GPU_PROFILE['tier']} | profile={GPU_PROFILE}")
print("Host Python:", sys.version.split()[0])
print(subprocess.check_output(["df", "-h", "/content"], text=True).strip())

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    print("Installing ffmpeg ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
print("ffmpeg:", shutil.which("ffmpeg"), "| ffprobe:", shutil.which("ffprobe"))

## Step 2 · 配置 Python 3.12 `uv venv` + `sglang-omni[fishaudio]` + FlashInfer AOT + 自动恢复 `sm_80` JIT 缓存

In [ ]:
import getpass
import glob
import hashlib
import io
import os
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
from pathlib import Path, PurePosixPath

# 推荐：从私有 HF Dataset 自动拉取已归档的 sm_80 JIT 缓存 (~16.34 MiB)，将冷启动从 ~24 分钟压至 ~5.5 分钟！
# 也支持 "drive"（Google Drive 本地路径）或 "none"（纯冷编译，不拉取缓存）。
JIT_CACHE_SOURCE = "hf"  #@param ["hf", "drive", "none"]
HF_JIT_DATASET_REPO = "exmeaning/moesekai-pjsk-tts-voices"  #@param {type:"string"}
HF_JIT_CACHE_FILE = "jit-cache/sglang-0.5.23-cu130-sm80.tar.gz"  #@param {type:"string"}
EXPECTED_JIT_SHA256 = "2a77f6f72f10d084cccde58742d6ccffcc3e490a2579e673a507e0b36ccb1006"
DRIVE_JIT_CACHE_TAR = ""  #@param {type:"string"}

SGLANG_REPO_DIR = Path("/content/sglang-omni")
VENV_DIR = Path("/content/sglang-venv")
VENV_PY = VENV_DIR / "bin" / "python"
SGL_OMNI_BIN = VENV_DIR / "bin" / "sgl-omni"
OVERRIDES_PATH = Path("/content/overrides.txt")
STUBS_DIR = Path("/content/stubs")

FLASHINFER_SHIM_WHL = (
    "https://github.com/flashinfer-ai/flashinfer/releases/download/v0.7.2/"
    "flashinfer_jit_cache-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl"
)
FLASHINFER_ARCH_WHEELS = {
    "8.0": "flashinfer_jit_cache_sm80-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl",
    "8.9": "flashinfer_jit_cache_sm89-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl",
    "9.0": "flashinfer_jit_cache_sm90a-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl",
    "10.0": "flashinfer_jit_cache_sm100a-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl",
    "10.3": "flashinfer_jit_cache_sm103a-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl",
    "12.0": "flashinfer_jit_cache_sm120f-0.7.2%2Bcu130-cp39-abi3-manylinux_2_28_x86_64.whl",
}

_RUNTIME_HF_TOKEN_CACHE = ""


def resolve_hf_token_runtime(*, optional: bool = False) -> str:
    """Resolve HF read token from env, Colab Secrets (`HF_TOKEN`), or interactive prompt (never logged/saved)."""
    global _RUNTIME_HF_TOKEN_CACHE
    if _RUNTIME_HF_TOKEN_CACHE:
        return _RUNTIME_HF_TOKEN_CACHE
    tok = (os.environ.get("HF_TOKEN") or "").strip()
    if not tok:
        try:
            from google.colab import userdata
            tok = (userdata.get("HF_TOKEN") or "").strip()
        except Exception:
            tok = ""
    if not tok:
        prompt = (
            "请输入私有 HF Dataset 读取 Token（留空则跳过 JIT 缓存回退冷编译，不会显示或落盘）: "
            if optional
            else "请输入具有私有数据集读取权限的 Hugging Face Token (不会显示或保存): "
        )
        tok = getpass.getpass(prompt).strip()
    if tok:
        _RUNTIME_HF_TOKEN_CACHE = tok
    elif not optional:
        raise RuntimeError("未提供 HF Token，无法拉取私有数据集文件。")
    return tok


t0 = time.monotonic()

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV_BIN = shutil.which("uv") or "uv"

if not SGLANG_REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/sgl-project/sglang-omni.git", str(SGLANG_REPO_DIR)],
        check=True,
    )
sglang_sha = subprocess.check_output(
    ["git", "-C", str(SGLANG_REPO_DIR), "rev-parse", "--short=12", "HEAD"], text=True
).strip()
print(f"sglang-omni HEAD: {sglang_sha}")

OVERRIDES_PATH.write_text("protobuf>=6.31.1,<7.0.0\n", encoding="utf-8")

if not VENV_PY.exists():
    subprocess.run([UV_BIN, "venv", "--python", "3.12", str(VENV_DIR)], check=True)

subprocess.run(
    [
        UV_BIN,
        "pip",
        "install",
        "--prerelease=allow",
        "--python",
        str(VENV_PY),
        "--override",
        str(OVERRIDES_PATH),
        "-e",
        f"{SGLANG_REPO_DIR}[fishaudio]",
        "hf_transfer",
        "ninja",
        "requests",
    ],
    cwd=str(SGLANG_REPO_DIR),
    check=True,
)

# 按实际 GPU 架构自动匹配 FlashInfer 0.7.2+cu130 shim + AOT 轮子（A100=sm80, H100=sm90a, G4/Pro6000=sm120f）
cc_str = str(GPU_INFO.get("compute_cap") or "8.0").strip()
arch_whl = FLASHINFER_ARCH_WHEELS.get(cc_str, FLASHINFER_ARCH_WHEELS["8.0"])
fi_wheels = [
    FLASHINFER_SHIM_WHL,
    f"https://github.com/flashinfer-ai/flashinfer/releases/download/v0.7.2/{arch_whl}",
]
subprocess.run(
    [UV_BIN, "pip", "install", "--python", str(VENV_PY), "--no-deps", *fi_wheels],
    check=True,
)

fi_chk = subprocess.check_output(
    [
        str(VENV_PY),
        "-c",
        (
            "import glob, os, flashinfer; "
            "from flashinfer.jit import env as fi_env; "
            "providers = [str(p.jit_cache_dir) for p in fi_env.FLASHINFER_AOT_PROVIDERS]; "
            "sos = sum(len(glob.glob(os.path.join(d, '**', '*.so'), recursive=True)) for d in providers); "
            "print(f'flashinfer={flashinfer.__version__} has_cache={fi_env.has_flashinfer_jit_cache()} aot_so_count={sos}')"
        ),
    ],
    text=True,
).strip()
print("FlashInfer AOT:", fi_chk)

# 准备加速环境变量、JIT 缓存目录与快速 Triton Autotuner sitecustomize
SERVE_ENV = dict(os.environ)
SERVE_ENV["PYTHONUNBUFFERED"] = "1"
SERVE_ENV["HF_HUB_DISABLE_XET"] = "1"
SERVE_ENV["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
SERVE_ENV["SGLANG_OMNI_STARTUP_TIMEOUT"] = "600"
SERVE_ENV["MAX_JOBS"] = "12"
SERVE_ENV["FLASHINFER_NVCC_THREADS"] = "4"
SERVE_ENV["SGLANG_JIT_CACHE_DIR"] = "/content/sglang_jit_cache"
SERVE_ENV["TRITON_CACHE_DIR"] = "/content/triton_cache"
SERVE_ENV["TVM_FFI_CACHE_DIR"] = "/content/tvm_ffi_cache"
SERVE_ENV["MOESEKAI_FAST_TRITON_AUTOTUNE"] = "1"

for d in ("/content/sglang_jit_cache", "/content/triton_cache", "/content/tvm_ffi_cache"):
    os.makedirs(d, exist_ok=True)

STUBS_DIR.mkdir(parents=True, exist_ok=True)
(STUBS_DIR / "sitecustomize.py").write_text(
    "import os\n"
    "if os.environ.get('MOESEKAI_FAST_TRITON_AUTOTUNE') == '1':\n"
    "    try:\n"
    "        import triton.runtime.autotuner as _ta\n"
    "        _orig_init = _ta.Autotuner.__init__\n"
    "        def _fast_init(self, fn, arg_names, configs, *args, **kwargs):\n"
    "            if isinstance(configs, (list, tuple)) and len(configs) > 2:\n"
    "                safe = [c for c in configs if getattr(c, 'num_stages', 2) <= 2 and getattr(c, 'num_warps', 4) in (2, 4)]\n"
    "                if not safe:\n"
    "                    safe = sorted(configs, key=lambda c: (getattr(c, 'num_stages', 2), getattr(c, 'num_warps', 4)))\n"
    "                configs = list(safe[:2])\n"
    "            return _orig_init(self, fn, arg_names, configs, *args, **kwargs)\n"
    "        _ta.Autotuner.__init__ = _fast_init\n"
    "    except Exception:\n"
    "        pass\n",
    encoding="utf-8",
)

SERVE_ENV["PATH"] = f"{VENV_DIR / 'bin'}:{SERVE_ENV.get('PATH', '')}"
SERVE_ENV["PYTHONPATH"] = str(STUBS_DIR) + (
    f":{SERVE_ENV['PYTHONPATH']}" if SERVE_ENV.get("PYTHONPATH") else ""
)

if os.path.exists("/usr/local/cuda/bin/nvcc"):
    SERVE_ENV["CUDA_HOME"] = "/usr/local/cuda"
    SERVE_ENV["PATH"] = f"/usr/local/cuda/bin:{SERVE_ENV['PATH']}"

# 补齐 venv 内 libcudart.so 软链（防止 nvcc 链接阶段缺失）
libdirs = sorted(
    {
        str(Path(p).parent)
        for p in glob.glob(str(VENV_DIR / "lib" / "python3.*" / "site-packages" / "nvidia" / "**" / "libcudart.so*"), recursive=True)
    }
)
for d in libdirs:
    for so in Path(d).glob("libcudart.so.*"):
        link = Path(d) / "libcudart.so"
        if not link.exists():
            try:
                link.symlink_to(so.name)
            except OSError:
                pass
if libdirs:
    joined = ":".join(libdirs)
    SERVE_ENV["LIBRARY_PATH"] = joined + (f":{SERVE_ENV['LIBRARY_PATH']}" if SERVE_ENV.get("LIBRARY_PATH") else "")
    SERVE_ENV["LD_LIBRARY_PATH"] = joined + (f":{SERVE_ENV['LD_LIBRARY_PATH']}" if SERVE_ENV.get("LD_LIBRARY_PATH") else "")

# 如缺少 deep_ep 则注入最小 stub
if subprocess.run([str(VENV_PY), "-c", "import deep_ep"], capture_output=True).returncode != 0:
    (STUBS_DIR / "deep_ep.py").write_text("class Buffer:\n    pass\nclass Config:\n    pass\n", encoding="utf-8")
    print("Added minimal deep_ep stub.")

# 安全解压 sm_80 预编译 JIT 缓存包（支持从私有 HF Dataset 或 Google Drive 恢复，带路径穿越防御与 SHA-256 校验）
ALLOWED_JIT_TOP = {"sglang_jit_cache", "triton_cache", "tvm_ffi_cache"}


def _safe_extract_jit_tar_bytes(tar_bytes: bytes, *, expected_sha256: str = "") -> int:
    actual_sha = hashlib.sha256(tar_bytes).hexdigest().lower()
    if expected_sha256 and actual_sha != expected_sha256.lower():
        raise ValueError(f"JIT cache SHA-256 mismatch: got {actual_sha}, expected {expected_sha256}")
    extract_root = Path("/content").resolve()
    with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode="r:gz") as tf:
        safe_members = []
        for m in tf.getmembers():
            raw = m.name or ""
            if raw in (".", "./") and m.isdir():
                continue
            if raw.startswith("/") or "\\" in raw or ":" in raw or ".." in PurePosixPath(raw).parts:
                raise ValueError(f"Unsafe tar member: {raw!r}")
            clean = raw[2:] if raw.startswith("./") else raw
            parts = PurePosixPath(clean).parts
            if not parts or parts[0] not in ALLOWED_JIT_TOP:
                continue
            if not (m.isfile() or m.isdir()) or m.issym() or m.islnk():
                raise ValueError(f"Disallowed tar member type: {raw!r}")
            target = (extract_root / Path(*parts)).resolve()
            if target != extract_root and extract_root not in target.parents:
                raise ValueError(f"Path escape in tar: {raw!r}")
            safe_members.append(m)
        tf.extractall(path=str(extract_root), members=safe_members, filter="data")
    return len(safe_members)


if JIT_CACHE_SOURCE == "hf" and cc_str == "8.0":
    try:
        tok = resolve_hf_token_runtime(optional=True)
        if tok:
            jit_url = f"https://huggingface.co/datasets/{HF_JIT_DATASET_REPO}/resolve/main/{HF_JIT_CACHE_FILE}"
            req = urllib.request.Request(
                jit_url,
                headers={"Authorization": f"Bearer {tok}", "User-Agent": "MoeSekai-TTS-Worker/1.0"},
            )
            t_jc = time.monotonic()
            with urllib.request.urlopen(req, timeout=120) as resp:
                jc_bytes = resp.read()
            restored_n = _safe_extract_jit_tar_bytes(jc_bytes, expected_sha256=EXPECTED_JIT_SHA256)
            print(
                f"Restored sm_80 JIT cache from HF ({HF_JIT_DATASET_REPO}/{HF_JIT_CACHE_FILE}): "
                f"{len(jc_bytes)/(1024**2):.2f} MiB, {restored_n} entries in {time.monotonic() - t_jc:.2f}s (SHA-256 OK!)"
            )
        else:
            print("未输入 HF Token，已跳过 HF JIT 缓存恢复，将走首次冷编译。")
    except Exception as jc_exc:
        print(f"WARN: HF JIT 缓存恢复失败（将自动回退冷编译）: {type(jc_exc).__name__}: {jc_exc}")
elif JIT_CACHE_SOURCE == "drive" and DRIVE_JIT_CACHE_TAR.strip():
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    tar_p = Path(DRIVE_JIT_CACHE_TAR.strip())
    if tar_p.exists():
        restored_n = _safe_extract_jit_tar_bytes(tar_p.read_bytes())
        print(f"Restored {restored_n} cached JIT entries from Drive: {tar_p}")
    else:
        print(f"Note: DRIVE_JIT_CACHE_TAR not found ({tar_p}); will run cold JIT compile.")
elif JIT_CACHE_SOURCE == "hf" and cc_str != "8.0":
    print(f"GPU compute_cap={cc_str} (non-sm_80); skipping sm_80 JIT tarball, FlashInfer {arch_whl} AOT active.")

print(f"Step 2 completed in {time.monotonic() - t0:.1f}s")

## Step 3 · 下载 `fishaudio/drama-3-int4` 权重（禁用 Xet 防 429 + 19 文件白名单 + `curl` LFS 回退）

In [ ]:
import subprocess
import time
from pathlib import Path

MODEL_ID = "fishaudio/drama-3-int4"
MODEL_DIR = Path("/content/models/drama-3-int4")
DRAMA3_ALLOW_PATTERNS = [
    "*.safetensors",
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
    "codec.pth",
]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.monotonic()

DL_HELPER = Path("/content/download_drama3.py")
DL_HELPER.write_text(
    """import fnmatch
import json
import os
import pathlib
import subprocess
import sys
import time
import urllib.request

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

model_id = sys.argv[1]
model_dir = sys.argv[2]
allow = json.loads(sys.argv[3])

ok = False
for attempt in range(1, 4):
    try:
        if attempt >= 2:
            os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
        hf_tr = os.environ.get("HF_HUB_ENABLE_HF_TRANSFER", "0")
        print(f"[download] attempt {attempt}/3 (HF_HUB_DISABLE_XET=1, hf_transfer={hf_tr})", flush=True)
        snapshot_download(repo_id=model_id, local_dir=model_dir, allow_patterns=allow, max_workers=2)
        ok = True
        break
    except Exception as exc:
        print(f"[download] attempt {attempt}/3 failed: {exc}", flush=True)
        if attempt < 3:
            time.sleep(10 * attempt)

if not ok:
    print("[download] Falling back to direct curl LFS download...", flush=True)
    req = urllib.request.Request(
        f"https://huggingface.co/api/models/{model_id}?blobs=true",
        headers={"User-Agent": "MoeSekai-TTS-Worker/1.0"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        siblings = json.loads(resp.read()).get("siblings", [])
    for item in siblings:
        fn = str(item.get("rfilename") or "")
        if not fn or not any(fnmatch.fnmatch(fn, pat) for pat in allow):
            continue
        exp_sz = int(item.get("size") or 0)
        dst = pathlib.Path(model_dir) / fn
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() and exp_sz > 0 and dst.stat().st_size == exp_sz:
            continue
        url = f"https://huggingface.co/{model_id}/resolve/main/{fn}"
        subprocess.run(
            ["curl", "-fsSL", "-C", "-", "--retry", "5", "--retry-delay", "5", "-A", "MoeSekai-TTS-Worker/1.0", "-o", str(dst), url],
            check=True,
        )
        print(f"[curl ok] {fn} ({dst.stat().st_size} B)", flush=True)
""",
    encoding="utf-8",
)

import json as _json
subprocess.run(
    [str(VENV_PY), str(DL_HELPER), MODEL_ID, str(MODEL_DIR), _json.dumps(DRAMA3_ALLOW_PATTERNS)],
    env=SERVE_ENV,
    check=True,
)

files = [p for p in MODEL_DIR.rglob("*") if p.is_file()]
total_gib = sum(p.stat().st_size for p in files) / (1024 ** 3)
dt = time.monotonic() - t0
print(f"Model ready at {MODEL_DIR}: {len(files)} files, {total_gib:.2f} GiB in {dt:.1f}s")

## Step 4 · 加载花里实乃理 (`pjsk_05_minori`) 官方干声参考音

支持 3 种零硬编码密钥加载方式：
- **`upload`**（默认推荐）：弹出文件选择器，直接上传本机 `<moesekai-tts>/.local_voices_out/voices/pjsk_05_minori/v1/ref.wav`
- **`drive`**：从已挂载 Google Drive 读取 `DRIVE_REF_WAV_PATH`
- **`hf`**：从私有 HF Dataset `exmeaning/moesekai-pjsk-tts-voices` 拉取（优先读取 Colab Secrets `HF_TOKEN`，否则运行时密文提示输入，绝不打印或落盘）

In [ ]:
import getpass
import hashlib
import json
import os
import shutil
import urllib.request
import wave
from pathlib import Path
from IPython.display import Audio, display

REF_SOURCE = "upload"  #@param ["upload", "drive", "hf"]
DRIVE_REF_WAV_PATH = "/content/drive/MyDrive/moesekai-tts/pjsk_05_minori/v1/ref.wav"  #@param {type:"string"}
HF_DATASET_REPO = "exmeaning/moesekai-pjsk-tts-voices"  #@param {type:"string"}
HF_REF_FILE = "voices/pjsk_05_minori/v1/ref.wav"  #@param {type:"string"}
HF_META_FILE = "voices/pjsk_05_minori/v1/meta.json"  #@param {type:"string"}
ALLOW_CUSTOM_REF_SHA256 = False  #@param {type:"boolean"}

EXPECTED_REF_SHA256 = "20fc0c53bc82015d81e8dd6668a211a82914f577c6bf51a2e8b879b17303bda5"
REF_TEXT = (
    "わたしの想いは、アイドルとしてみんなに希望を届けたい、だからなあ "
    "えっとね、警察のお手伝いで地域の皆さんに安全を呼びかけるお仕事なんだって "
    "人気のアイドルがよくやってるから、ちょっと憧れてたなあ"
)

REFS_DIR = Path("/content/refs")
REFS_DIR.mkdir(parents=True, exist_ok=True)
REF_WAV_PATH = REFS_DIR / "pjsk_05_minori_ref.wav"


def _resolve_hf_token_runtime() -> str:
    if "resolve_hf_token_runtime" in globals():
        return resolve_hf_token_runtime(optional=False)
    tok = (os.environ.get("HF_TOKEN") or "").strip()
    if tok:
        return tok
    try:
        from google.colab import userdata
        tok = (userdata.get("HF_TOKEN") or "").strip()
        if tok:
            return tok
    except Exception:
        pass
    tok = getpass.getpass("请输入具有私有数据集读取权限的 Hugging Face Token (不会显示或保存): ").strip()
    if not tok:
        raise RuntimeError("未提供 HF Token，无法从私有数据集拉取 ref.wav。")
    return tok


if REF_SOURCE == "upload":
    from google.colab import files as colab_files
    print("请选择本机参考干声：<moesekai-tts>/.local_voices_out/voices/pjsk_05_minori/v1/ref.wav")
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError("未上传任何文件，请重新运行此单元格并上传 ref.wav。")
    wav_name = next((k for k in uploaded if k.lower().endswith(".wav")), next(iter(uploaded)))
    REF_WAV_PATH.write_bytes(uploaded[wav_name])
    if "meta.json" in uploaded:
        meta_doc = json.loads(uploaded["meta.json"].decode("utf-8"))
        if meta_doc.get("ref_text"):
            REF_TEXT = str(meta_doc["ref_text"]).strip()
    print(f"Uploaded reference file: {wav_name} -> {REF_WAV_PATH}")

elif REF_SOURCE == "drive":
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    src = Path(DRIVE_REF_WAV_PATH)
    if not src.exists():
        raise FileNotFoundError(f"Google Drive 中未找到参考干声：{src}")
    shutil.copy2(src, REF_WAV_PATH)
    meta_sibling = src.parent / "meta.json"
    if meta_sibling.exists():
        meta_doc = json.loads(meta_sibling.read_text(encoding="utf-8"))
        if meta_doc.get("ref_text"):
            REF_TEXT = str(meta_doc["ref_text"]).strip()
    print(f"Loaded reference from Drive: {src}")

elif REF_SOURCE == "hf":
    hf_token = _resolve_hf_token_runtime()
    headers = {
        "Authorization": f"Bearer {hf_token}",
        "User-Agent": "MoeSekai-TTS-Worker/1.0",
    }
    wav_url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/{HF_REF_FILE}"
    req = urllib.request.Request(wav_url, headers=headers)
    with urllib.request.urlopen(req, timeout=120) as resp:
        REF_WAV_PATH.write_bytes(resp.read())
    try:
        meta_url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/{HF_META_FILE}"
        with urllib.request.urlopen(urllib.request.Request(meta_url, headers=headers), timeout=30) as mresp:
            meta_doc = json.loads(mresp.read().decode("utf-8"))
            if meta_doc.get("ref_text"):
                REF_TEXT = str(meta_doc["ref_text"]).strip()
    except Exception:
        pass
    del hf_token
    print(f"Downloaded private reference from HF dataset: {HF_DATASET_REPO}/{HF_REF_FILE}")
else:
    raise ValueError(f"Unsupported REF_SOURCE: {REF_SOURCE!r}")

raw_bytes = REF_WAV_PATH.read_bytes()
actual_sha = hashlib.sha256(raw_bytes).hexdigest()
if actual_sha != EXPECTED_REF_SHA256:
    msg = (
        f"ref.wav SHA-256 与标准 pjsk_05_minori/v1 不一致：\n"
        f"  actual  : {actual_sha}\n"
        f"  expected: {EXPECTED_REF_SHA256}"
    )
    if not ALLOW_CUSTOM_REF_SHA256:
        raise ValueError(msg + "\n若您确认使用了自定义干声，请勾选 ALLOW_CUSTOM_REF_SHA256 后重试。")
    print("WARN:", msg)
else:
    print(f"SHA-256 VERIFIED OK: {actual_sha[:16]}... ({len(raw_bytes):,} bytes)")

with wave.open(str(REF_WAV_PATH), "rb") as wf:
    sr = wf.getframerate()
    ch = wf.getnchannels()
    dur = wf.getnframes() / float(sr)
print(f"Reference WAV: {sr} Hz, {ch} ch, {dur:.3f}s")
print(f"Reference Text: {REF_TEXT}")
display(Audio(filename=str(REF_WAV_PATH)))

## Step 5 · 加载 8 幕 / 16 句花里实乃理 PV 配音脚本（与 `promo/voiceover/script.json` 1:1 同步）

In [ ]:
import json
import re
from pathlib import Path

UPLOAD_CUSTOM_SCRIPT_JSON = False  #@param {type:"boolean"}

EMBEDDED_SCRIPT_JSON = r'''
{
  "schema": "moesekai-pv-voiceover/1",
  "project": "MoeSekai Story Studio Bilibili PV (Keynote Edition)",
  "studio_version": "0.1.4",
  "language": "ja-JP",
  "subtitle_languages": [
    "zh-CN",
    "ja-JP"
  ],
  "model": "fishaudio/drama-3-int4",
  "engine": "sglang-omni",
  "voice": {
    "voice_id": "pjsk_05_minori",
    "voice_ver": 1,
    "unit": "mmj",
    "display_name_ja": "花里みのり",
    "display_name_zh": "花里实乃理",
    "display_name_en": "Hanasato Minori",
    "ref_sha256": "20fc0c53bc82015d81e8dd6668a211a82914f577c6bf51a2e8b879b17303bda5",
    "ref_duration_sec": 15.981,
    "ref_sample_rate": 44100,
    "ref_lufs": -16.0,
    "ref_text": "わたしの想いは、アイドルとしてみんなに希望を届けたい、だからなあ えっとね、警察のお手伝いで地域の皆さんに安全を呼びかけるお仕事なんだって 人気のアイドルがよくやってるから、ちょっと憧れてたなあ",
    "local_ref_path_hint": "<moesekai-tts>/.local_voices_out/voices/pjsk_05_minori/v1/ref.wav",
    "hf_private_dataset": "exmeaning/moesekai-pjsk-tts-voices",
    "hf_ref_path": "voices/pjsk_05_minori/v1/ref.wav",
    "hf_meta_path": "voices/pjsk_05_minori/v1/meta.json",
    "hf_jit_cache_path": "jit-cache/sglang-0.5.23-cu130-sm80.tar.gz",
    "hf_jit_cache_sha256": "2a77f6f72f10d084cccde58742d6ccffcc3e490a2579e673a507e0b36ccb1006"
  },
  "audio_spec": {
    "format": "wav",
    "codec": "pcm_s16le",
    "sample_rate": 48000,
    "channels": 1,
    "target_lufs": -16.0,
    "true_peak_db": -1.5,
    "lra": 11.0,
    "ffmpeg_filter_chain": "silenceremove=start_periods=1:start_silence=0.1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_silence=0.15:start_threshold=-45dB,areverse,loudnorm=I=-16:TP=-1.5:LRA=11",
    "note": "Lossless 48kHz mono 16-bit PCM WAV for Bilibili PV mastering (NOT 40kbps Opus)."
  },
  "drama3_api_contract": {
    "endpoint": "http://127.0.0.1:8000/v1/audio/speech",
    "allowed_fields": [
      "input",
      "response_format",
      "max_new_tokens",
      "temperature",
      "top_p",
      "references"
    ],
    "forbidden_fields": [
      "language",
      "instructions",
      "seed",
      "voice"
    ],
    "default_temperature": 1.0,
    "default_top_p": 0.9,
    "min_vram_gib": 32,
    "allowed_gpus": [
      "A100-80GB",
      "A100-40GB"
    ],
    "forbidden_gpus": [
      "L4",
      "T4",
      "V100",
      "P100",
      "K80",
      "P4"
    ]
  },
  "timing_defaults": {
    "inter_line_gap_sec": 0.25,
    "scene_tail_pad_sec": 0.45,
    "final_scene_tail_pad_sec": 0.9,
    "estimated_total_duration_sec": 85.0
  },
  "scenes": [
    {
      "scene_id": "s01_hook",
      "index": 1,
      "title_zh": "从一个脑洞开始",
      "title_ja": "ひとつのアイデアから始まる",
      "target_duration_sec": 7.72,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "01_hook_a",
        "02_hook_b"
      ],
      "visual_cue_zh": "纯净浅色 Keynote 开场：展示脑海中的 PJSK 原创剧情与传统抠立绘、对口型、剪时间轴的门槛痛点"
    },
    {
      "scene_id": "s02_wish_agent",
      "index": 2,
      "title_zh": "许个愿，就能生成整幕剧情",
      "title_ja": "願いを伝えるだけで脚本が完成",
      "target_duration_sec": 12.48,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "03_wish_a",
        "04_wish_b"
      ],
      "visual_cue_zh": "核心主角幕：输入一句愿望 → AI 检索官方设定 → 编写 .msk 剧本 → 自动静态校验 0 errors 与全分支推演 → 快照一键回滚（实机 06 + 07 特写）"
    },
    {
      "scene_id": "s03_live2d",
      "index": 3,
      "title_zh": "Live2D 多角色舞台实时开演",
      "title_ja": "Live2D マルチアクター即時再生",
      "target_duration_sec": 11.33,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "05_live2d_a",
        "06_live2d_b"
      ],
      "visual_cue_zh": "实机 60fps 录屏 clip-01 与 clip-02：多位 Live2D 角色带动作表情与服装同台演出，点击分支选项立刻进入不同剧情走向"
    },
    {
      "scene_id": "s04_flow_msk",
      "index": 4,
      "title_zh": "节点画布与 .msk 实时双向同步",
      "title_ja": "ノードキャンバス＆.msk 双方向同期",
      "target_duration_sec": 11.0,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "07_flow_a",
        "08_msk_b"
      ],
      "visual_cue_zh": "实机 clip-03 与 clip-04：在可视化节点画布上拖拽连线梳理多结局分支，并在 .msk 纯文本视图精修对白，双向同步即刻生效"
    },
    {
      "scene_id": "s05_assets_cast",
      "index": 5,
      "title_zh": "内置 26 角色素材与演出检视器",
      "title_ja": "26キャラ素材＆演出インスペクター",
      "target_duration_sec": 10.67,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "09_assets_a",
        "10_assets_b"
      ],
      "visual_cue_zh": "实机 08 与 09 截图聚焦：直接浏览 26 位角色服装、动作、表情与舞台背景，在演出检视器中为每句对白点选表演与 BGM"
    },
    {
      "scene_id": "s06_i18n_voice",
      "index": 6,
      "title_zh": "多语言字幕校对与自定义配音打包",
      "title_ja": "多言語校閲＆カスタムボイス梱包",
      "target_duration_sec": 10.01,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "11_i18n_a",
        "12_voice_b"
      ],
      "visual_cue_zh": "实机 10、11、12 截图聚焦：5 语言本地化双语校对界面 + 导入自定义配音/TTS 语音清单自动匹配对白并一键打包"
    },
    {
      "scene_id": "s07_preflight_open",
      "index": 7,
      "title_zh": "导出自动体检，全平台开源免费",
      "title_ja": "書き出し前自動検査＆無料オープンソース",
      "target_duration_sec": 10.83,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.45,
      "line_ids": [
        "13_preflight_a",
        "14_open_b"
      ],
      "visual_cue_zh": "实机 13 预检弹窗（0 errors, 0 warnings）与 01/16 全平台桌面展示，与 pjsk.moe 网页端互通，Win/Mac/Linux MPL-2.0 开源免费"
    },
    {
      "scene_id": "s08_cta",
      "index": 8,
      "title_zh": "前往 story.pjsk.moe 把愿望搬上舞台",
      "title_ja": "story.pjsk.moe から始めよう＆三連お願い",
      "target_duration_sec": 10.96,
      "inter_line_gap_sec": 0.25,
      "post_scene_pad_sec": 0.9,
      "line_ids": [
        "15_cta_a",
        "16_cta_b"
      ],
      "visual_cue_zh": "纯白 Keynote 终幕：story.pjsk.moe 清晰主视觉 + 桌面端下载入口 + B 站一键三连收尾动效"
    }
  ],
  "lines": [
    {
      "id": "01_hook_a",
      "index": 1,
      "scene_id": "s01_hook",
      "speaker": "pjsk_05_minori",
      "emotion": "cheerful, curious, bright smile",
      "text_tts": "[cheerful, curious, bright smile] みんなー！オリジナル物語、動かしてみたいよね？",
      "subtitle_ja": "みんなー！オリジナル物語、動かしてみたいよね？",
      "subtitle_zh": "大家——！想把脑海里的原创剧情直接搬上舞台吗？",
      "spoken_chars": 23,
      "max_new_tokens": 260,
      "est_duration_sec": 3.92,
      "audio_file": "audio/01_hook_a.wav",
      "audio_filename": "01_hook_a.wav",
      "guide_audio_file": "guide_audio/01_hook_a.wav"
    },
    {
      "id": "02_hook_b",
      "index": 2,
      "scene_id": "s01_hook",
      "speaker": "pjsk_05_minori",
      "emotion": "lightly troubled, relatable sigh",
      "text_tts": "[lightly troubled, relatable sigh] でも切り抜きや口パク編集って大変……",
      "subtitle_ja": "でも切り抜きや口パク編集って大変……",
      "subtitle_zh": "可抠立绘、对口型、剪时间轴实在太劝退了……",
      "spoken_chars": 18,
      "max_new_tokens": 260,
      "est_duration_sec": 3.1,
      "audio_file": "audio/02_hook_b.wav",
      "audio_filename": "02_hook_b.wav",
      "guide_audio_file": "guide_audio/02_hook_b.wav"
    },
    {
      "id": "03_wish_a",
      "index": 3,
      "scene_id": "s02_wish_agent",
      "speaker": "pjsk_05_minori",
      "emotion": "excited, bright smile",
      "text_tts": "[excited, bright smile] モエセカイ・ストーリースタジオなら、願いを伝えるだけで脚本が完成！",
      "subtitle_ja": "MoeSekai Story Studio なら、願いを伝えるだけで脚本が完成！",
      "subtitle_zh": "在 MoeSekai Story Studio，许个愿就能生成整幕剧情！",
      "spoken_chars": 33,
      "max_new_tokens": 280,
      "est_duration_sec": 5.56,
      "audio_file": "audio/03_wish_a.wav",
      "audio_filename": "03_wish_a.wav",
      "guide_audio_file": "guide_audio/03_wish_a.wav"
    },
    {
      "id": "04_wish_b",
      "index": 4,
      "scene_id": "s02_wish_agent",
      "speaker": "pjsk_05_minori",
      "emotion": "earnest, lively, reassuring",
      "text_tts": "[earnest, lively, reassuring] エーアイが公式設定を調べて分岐を検証、スナップショットでいつでも戻せるよ！",
      "subtitle_ja": "AIが公式設定を調べて分岐を検証、スナップショットでいつでも戻せるよ！",
      "subtitle_zh": "内置 AI 检索官方设定、自动校验全分支，每一步快照随时回滚！",
      "spoken_chars": 37,
      "max_new_tokens": 310,
      "est_duration_sec": 6.22,
      "audio_file": "audio/04_wish_b.wav",
      "audio_filename": "04_wish_b.wav",
      "guide_audio_file": "guide_audio/04_wish_b.wav"
    },
    {
      "id": "05_live2d_a",
      "index": 5,
      "scene_id": "s03_live2d",
      "speaker": "pjsk_05_minori",
      "emotion": "amazed, sparkling with excitement",
      "text_tts": "[amazed, sparkling with excitement] 脚本はすぐ再生！ライブツーディーの仲間が衣装や表情つきで動くよ！",
      "subtitle_ja": "脚本はすぐ再生！Live2Dの仲間が衣装や表情つきで動くよ！",
      "subtitle_zh": "剧本生成即可运行，多位 Live2D 角色带动作表情与服装同台演绎！",
      "spoken_chars": 32,
      "max_new_tokens": 280,
      "est_duration_sec": 5.4,
      "audio_file": "audio/05_live2d_a.wav",
      "audio_filename": "05_live2d_a.wav",
      "guide_audio_file": "guide_audio/05_live2d_a.wav"
    },
    {
      "id": "06_live2d_b",
      "index": 6,
      "scene_id": "s03_live2d",
      "speaker": "pjsk_05_minori",
      "emotion": "happy, upbeat, inviting",
      "text_tts": "[happy, upbeat, inviting] 立ち位置もカメラも確認できて、選択肢で別ルートもすぐ遊べるの！",
      "subtitle_ja": "立ち位置もカメラも確認できて、選択肢で別ルートもすぐ遊べるの！",
      "subtitle_zh": "站位与镜头实时呈现，内置分支播放器点击选项立刻体验不同走向！",
      "spoken_chars": 31,
      "max_new_tokens": 280,
      "est_duration_sec": 5.23,
      "audio_file": "audio/06_live2d_b.wav",
      "audio_filename": "06_live2d_b.wav",
      "guide_audio_file": "guide_audio/06_live2d_b.wav"
    },
    {
      "id": "07_flow_a",
      "index": 7,
      "scene_id": "s04_flow_msk",
      "speaker": "pjsk_05_minori",
      "emotion": "bright, explaining clearly",
      "text_tts": "[bright, explaining clearly] こだわりたいときは、ノードキャンバスで線をつないで分岐を整理！",
      "subtitle_ja": "こだわりたいときは、ノードキャンバスで線をつないで分岐を整理！",
      "subtitle_zh": "想亲手精修？在可视化节点画布上拖拽连线梳理多结局分支！",
      "spoken_chars": 31,
      "max_new_tokens": 280,
      "est_duration_sec": 5.4,
      "audio_file": "audio/07_flow_a.wav",
      "audio_filename": "07_flow_a.wav",
      "guide_audio_file": "guide_audio/07_flow_a.wav"
    },
    {
      "id": "08_msk_b",
      "index": 8,
      "scene_id": "s04_flow_msk",
      "speaker": "pjsk_05_minori",
      "emotion": "impressed, clear, upbeat",
      "text_tts": "[impressed, clear, upbeat] エムエスケー画面とも双方向同期、直した瞬間にすぐ反映だよ！",
      "subtitle_ja": ".msk テキスト画面とも双方向同期、直した瞬間にすぐ反映だよ！",
      "subtitle_zh": "与 .msk 纯文本视图实时双向同步，画布或代码修改即刻生效！",
      "spoken_chars": 29,
      "max_new_tokens": 260,
      "est_duration_sec": 4.9,
      "audio_file": "audio/08_msk_b.wav",
      "audio_filename": "08_msk_b.wav",
      "guide_audio_file": "guide_audio/08_msk_b.wav"
    },
    {
      "id": "09_assets_a",
      "index": 9,
      "scene_id": "s05_assets_cast",
      "speaker": "pjsk_05_minori",
      "emotion": "cheerful, proud, enthusiastic",
      "text_tts": "[cheerful, proud, enthusiastic] 二十六人の衣装・動き・背景を収録、素材探しはいらないよ！",
      "subtitle_ja": "26人の衣装・モーション・背景を収録、素材探しはいらないよ！",
      "subtitle_zh": "内置 26 位角色服装、动作、表情与舞台背景，不用手动找图抠图！",
      "spoken_chars": 28,
      "max_new_tokens": 260,
      "est_duration_sec": 4.74,
      "audio_file": "audio/09_assets_a.wav",
      "audio_filename": "09_assets_a.wav",
      "guide_audio_file": "guide_audio/09_assets_a.wav"
    },
    {
      "id": "10_assets_b",
      "index": 10,
      "scene_id": "s05_assets_cast",
      "speaker": "pjsk_05_minori",
      "emotion": "warm, focused, friendly",
      "text_tts": "[warm, focused, friendly] インスペクターで選ぶだけで、セリフごとの演技や曲も組めるんだ！",
      "subtitle_ja": "インスペクターで選ぶだけで、セリフごとの演技やBGMも組めるんだ！",
      "subtitle_zh": "在演出检视器中直接点选，轻松为每句对白编排专属表演与 BGM！",
      "spoken_chars": 31,
      "max_new_tokens": 280,
      "est_duration_sec": 5.23,
      "audio_file": "audio/10_assets_b.wav",
      "audio_filename": "10_assets_b.wav",
      "guide_audio_file": "guide_audio/10_assets_b.wav"
    },
    {
      "id": "11_i18n_a",
      "index": 11,
      "scene_id": "s06_i18n_voice",
      "speaker": "pjsk_05_minori",
      "emotion": "warm, upbeat, clear",
      "text_tts": "[warm, upbeat, clear] 五つの言語パック対応、字幕の対訳校閲も一画面で完結！",
      "subtitle_ja": "5言語パック対応、字幕の対訳校閲も1画面で完結！",
      "subtitle_zh": "内置 5 语言本地化双语校对界面，同一视图完成多语字幕校对！",
      "spoken_chars": 26,
      "max_new_tokens": 260,
      "est_duration_sec": 4.41,
      "audio_file": "audio/11_i18n_a.wav",
      "audio_filename": "11_i18n_a.wav",
      "guide_audio_file": "guide_audio/11_i18n_a.wav"
    },
    {
      "id": "12_voice_b",
      "index": 12,
      "scene_id": "s06_i18n_voice",
      "speaker": "pjsk_05_minori",
      "emotion": "cheerful, excited",
      "text_tts": "[cheerful, excited] ボイスや音声リストを入れれば、自動対応して一括パッケージ！",
      "subtitle_ja": "ボイスやTTS音声リストを入れれば、自動対応して一括パッケージ！",
      "subtitle_zh": "导入自有配音或 TTS 语音清单，自动匹配对白并一键打包完整作品！",
      "spoken_chars": 29,
      "max_new_tokens": 260,
      "est_duration_sec": 4.9,
      "audio_file": "audio/12_voice_b.wav",
      "audio_filename": "12_voice_b.wav",
      "guide_audio_file": "guide_audio/12_voice_b.wav"
    },
    {
      "id": "13_preflight_a",
      "index": 13,
      "scene_id": "s07_preflight_open",
      "speaker": "pjsk_05_minori",
      "emotion": "reassuring, clear, helpful",
      "text_tts": "[reassuring, clear, helpful] 書き出し前に素材抜けや分岐エラーを自動検査して安全に出力！",
      "subtitle_ja": "書き出し前に素材抜けや分岐エラーを自動検査して出力！",
      "subtitle_zh": "导出前自动排查缺失素材与断链分支，一键导出 .msk 或独立资源包！",
      "spoken_chars": 29,
      "max_new_tokens": 260,
      "est_duration_sec": 4.9,
      "audio_file": "audio/13_preflight_a.wav",
      "audio_filename": "13_preflight_a.wav",
      "guide_audio_file": "guide_audio/13_preflight_a.wav"
    },
    {
      "id": "14_open_b",
      "index": 14,
      "scene_id": "s07_preflight_open",
      "speaker": "pjsk_05_minori",
      "emotion": "bright, warm, generous",
      "text_tts": "[bright, warm, generous] ウェブ版とも互換、ウィンドウズもマックも無料のオープンソース！",
      "subtitle_ja": "pjsk.moe Web版互換！Windows・macOS・Linux 対応の無料オープンソース！",
      "subtitle_zh": "与 pjsk.moe 网页端互通，全平台覆盖且 MPL-2.0 开源免费！",
      "spoken_chars": 31,
      "max_new_tokens": 280,
      "est_duration_sec": 5.23,
      "audio_file": "audio/14_open_b.wav",
      "audio_filename": "14_open_b.wav",
      "guide_audio_file": "guide_audio/14_open_b.wav"
    },
    {
      "id": "15_cta_a",
      "index": 15,
      "scene_id": "s08_cta",
      "speaker": "pjsk_05_minori",
      "emotion": "enthusiastic, warm, inviting",
      "text_tts": "[enthusiastic, warm, inviting] さあ公式サイトから体験して、あなたの願いを本物のステージに！",
      "subtitle_ja": "さあ story.pjsk.moe から体験して、あなたの願いを本物のステージに！",
      "subtitle_zh": "前往 story.pjsk.moe 下载体验，把你的愿望变成真实的舞台！",
      "spoken_chars": 30,
      "max_new_tokens": 280,
      "est_duration_sec": 5.07,
      "audio_file": "audio/15_cta_a.wav",
      "audio_filename": "15_cta_a.wav",
      "guide_audio_file": "guide_audio/15_cta_a.wav"
    },
    {
      "id": "16_cta_b",
      "index": 16,
      "scene_id": "s08_cta",
      "speaker": "pjsk_05_minori",
      "emotion": "joyful, sparkling idol energy, laughs softly",
      "text_tts": "[joyful, sparkling idol energy] よかったら、[laughs softly] いいね、コイン、お気に入りの三連もお願いね！",
      "subtitle_ja": "よかったら、いいね・コイン・お気に入りの三連もお願いね！",
      "subtitle_zh": "喜欢的话别忘了点赞、投币、收藏一键三连支持我们！",
      "spoken_chars": 28,
      "max_new_tokens": 280,
      "est_duration_sec": 4.74,
      "audio_file": "audio/16_cta_b.wav",
      "audio_filename": "16_cta_b.wav",
      "guide_audio_file": "guide_audio/16_cta_b.wav"
    }
  ],
  "ab_comparison_samples": [
    {
      "id": "ab_01_wish",
      "label": "核心卖点：许愿生成剧情 (JA 原生 vs ZH 跨语种)",
      "ja_input": "[excited, bright smile] モエセカイ・ストーリースタジオなら、願いを伝えるだけで脚本が完成！",
      "zh_input": "[excited, bright smile] 在萌世界剧情工坊，只要像许愿一样说出想法，AI 就能自动生成整幕剧本！"
    },
    {
      "id": "ab_02_live2d",
      "label": "Live2D 舞台实时开演 (JA 原生 vs ZH 跨语种)",
      "ja_input": "[amazed, sparkling with excitement] 脚本はすぐ再生！ライブツーディーの仲間が衣装や表情つきで動くよ！",
      "zh_input": "[amazed, sparkling with excitement] 剧本生成立刻开演，多位角色带着服装和表情在舞台上实时动起来！"
    },
    {
      "id": "ab_03_cta",
      "label": "B站三连收尾 (JA 原生 vs ZH 跨语种)",
      "ja_input": "[joyful, sparkling idol energy] よかったら、[laughs softly] いいね、コイン、お気に入りの三連もお願いね！",
      "zh_input": "[joyful, sparkling idol energy] 喜欢的话，[laughs softly] 别忘了点赞、投币、收藏一键三连支持我们！"
    }
  ]
}
'''

PV_SCRIPT = json.loads(EMBEDDED_SCRIPT_JSON)

if UPLOAD_CUSTOM_SCRIPT_JSON:
    from google.colab import files as colab_files
    print("请上传自定义的 promo/voiceover/script.json：")
    up = colab_files.upload()
    if up:
        first_key = next(iter(up))
        PV_SCRIPT = json.loads(up[first_key].decode("utf-8"))
        print(f"Loaded custom script from {first_key}")

LINES = PV_SCRIPT["lines"]
SCENES = PV_SCRIPT["scenes"]
AB_SAMPLES = PV_SCRIPT.get("ab_comparison_samples", [])

# 校验脚本字段与情绪标签格式（严禁中文动作括号）
CN_BRACKET_RE = re.compile(r"[【】（()）]")
for ln in LINES:
    for req_key in ("id", "scene_id", "text_tts", "subtitle_ja", "subtitle_zh", "max_new_tokens"):
        if req_key not in ln:
            raise ValueError(f"Line missing required field {req_key}: {ln}")
    if CN_BRACKET_RE.search(ln["text_tts"]):
        raise ValueError(f"Found disallowed Chinese/parenthesis stage bracket in {ln['id']}: {ln['text_tts']}")

print(f"Loaded {len(SCENES)} scenes / {len(LINES)} lines (est. {PV_SCRIPT['timing_defaults']['estimated_total_duration_sec']:.2f}s)")
for ln in LINES:
    print(f"  {ln['id']:11s} | {ln['scene_id']:14s} | {ln['spoken_chars']:2d}c | {ln['subtitle_zh']}")

## Step 6 · 启动本地 `sgl-omni serve` 并初始化严格 Schema 合成客户端与 48 kHz 无损后处理链

In [ ]:
import io
import json
import os
import re
import subprocess
import time
import urllib.error
import urllib.request
import wave
from pathlib import Path

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
SERVER_BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
SERVER_LOG_PATH = Path("/content/sgl_omni_serve.log")

ALLOWED_SPEECH_KEYS = {"input", "response_format", "max_new_tokens", "temperature", "top_p", "references"}
FORBIDDEN_SPEECH_KEYS = {"language", "instructions", "seed", "voice"}
AUDIO_FILTER_CHAIN = (
    "silenceremove=start_periods=1:start_silence=0.1:start_threshold=-45dB,"
    "areverse,"
    "silenceremove=start_periods=1:start_silence=0.15:start_threshold=-45dB,"
    "areverse,"
    "loudnorm=I=-16:TP=-1.5:LRA=11"
)


def stop_sgl_omni_server() -> None:
    proc = globals().get("SERVER_PROC")
    if proc is not None:
        try:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait(timeout=5)
        except Exception:
            pass
        globals()["SERVER_PROC"] = None


def build_serve_command() -> list[str]:
    cmd = [
        str(SGL_OMNI_BIN),
        "serve",
        "--model-path",
        str(MODEL_DIR),
        "--host",
        SERVER_HOST,
        "--port",
        str(SERVER_PORT),
        "--tts_engine.gpu_memory_fraction",
        f"{GPU_PROFILE['tts_engine_gpu_memory_fraction']:.2f}",
        "--vocoder.gpu_memory_fraction",
        f"{GPU_PROFILE['vocoder_gpu_memory_fraction']:.2f}",
        "--vocoder.factory.codec_path",
        str(MODEL_DIR),
        "--tts_engine.engine.disable_cuda_graph",
        "true",
        "--tts_engine.engine.cuda_graph_backend_prefill",
        "disabled",
        "--tts_engine.factory.gdn_prefill_cuda_graphs",
        "false",
        "--tts_engine.factory.first_frame_cuda_graphs",
        "false",
        "--vocoder.factory.window_cuda_graphs",
        "false",
        "--preprocessing.factory.reference_encode_graph_max_seconds",
        "0",
    ]
    if GPU_PROFILE["preprocessing_total_reserve_bytes"]:
        cmd.extend(["--preprocessing.total_reserve_bytes", GPU_PROFILE["preprocessing_total_reserve_bytes"]])
    return cmd


stop_sgl_omni_server()
SERVE_CMD = build_serve_command()
print("Launching:", " ".join(SERVE_CMD))
t_boot = time.monotonic()
_log_fh = open(SERVER_LOG_PATH, "w", encoding="utf-8")
SERVER_PROC = subprocess.Popen(SERVE_CMD, stdout=_log_fh, stderr=subprocess.STDOUT, env=SERVE_ENV)

boot_deadline = time.monotonic() + 600.0
last_pos = 0
ready = False
while time.monotonic() < boot_deadline:
    rc = SERVER_PROC.poll()
    if rc is not None:
        _log_fh.flush()
        tail = SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(f"sgl-omni serve exited early (rc={rc})!\n{tail}")
    try:
        txt = SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > last_pos:
            for ln in txt[last_pos:].splitlines():
                ls = ln.strip()
                if ls and any(k in ls.lower() for k in ("load weight", "kv cache", "mamba", "vocoder", "ready", "uvicorn", "error", "traceback")):
                    print("  [srv]", ls[:220], flush=True)
            last_pos = len(txt)
    except Exception:
        pass
    try:
        with urllib.request.urlopen(f"{SERVER_BASE_URL}/health", timeout=3.0) as r:
            if 200 <= r.status < 300:
                ready = True
                break
    except Exception:
        pass
    time.sleep(3.0)

if not ready:
    stop_sgl_omni_server()
    tail = SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-3000:]
    raise TimeoutError(f"sgl-omni serve did not become healthy within 600s!\n{tail}")

gpu_boot = query_gpu_info()
print(
    f"sgl-omni serve READY in {time.monotonic() - t_boot:.1f}s | "
    f"VRAM={gpu_boot['vram_used_mib']}/{gpu_boot['vram_total_mib']} MiB"
)


def count_spoken_chars(text: str) -> int:
    s = re.sub(r"\[[^\]]*\]", "", text or "")
    s = re.sub(r"\s+", "", s)
    return len(s)


def raw_wav_duration_sec(wav_bytes: bytes) -> float:
    with wave.open(io.BytesIO(wav_bytes), "rb") as wf:
        sr = wf.getframerate()
        return wf.getnframes() / float(sr) if sr else 0.0


def synthesize_drama3_wav(
    *,
    text: str,
    ref_audio_path: Path,
    ref_text: str,
    max_new_tokens: int = 350,
    temperature: float = 1.0,
    top_p: float = 0.9,
    timeout_sec: float = 180.0,
) -> bytes:
    """Call local sgl-omni /v1/audio/speech using strictly the 6 live-verified JSON fields."""
    payload = {
        "input": text,
        "response_format": "wav",
        "max_new_tokens": int(max_new_tokens),
        "temperature": float(temperature),
        "top_p": float(top_p),
        "references": [
            {
                "audio_path": str(ref_audio_path),
                "text": ref_text,
            }
        ],
    }
    assert set(payload.keys()) == ALLOWED_SPEECH_KEYS
    assert not (set(payload.keys()) & FORBIDDEN_SPEECH_KEYS)

    req = urllib.request.Request(
        f"{SERVER_BASE_URL}/v1/audio/speech",
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        method="POST",
        headers={"Content-Type": "application/json", "Accept": "audio/wav"},
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout_sec) as resp:
            body = resp.read()
    except urllib.error.HTTPError as he:
        err = he.read().decode("utf-8", errors="replace")[:500]
        raise RuntimeError(f"/v1/audio/speech HTTP {he.code}: {err}") from he
    if not body:
        raise RuntimeError("/v1/audio/speech returned empty audio body")
    return body


def postprocess_wav_pcm48k(raw_wav_path: Path, out_wav_path: Path) -> dict:
    """Trim leading/trailing silence + -16 LUFS loudnorm -> lossless 48kHz mono 16-bit PCM WAV."""
    out_wav_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "ffmpeg",
        "-y",
        "-hide_banner",
        "-loglevel",
        "error",
        "-i",
        str(raw_wav_path),
        "-af",
        AUDIO_FILTER_CHAIN,
        "-ac",
        "1",
        "-ar",
        "48000",
        "-c:a",
        "pcm_s16le",
        str(out_wav_path),
    ]
    subprocess.run(cmd, check=True, capture_output=True, text=True, timeout=60)
    dur_str = subprocess.check_output(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            str(out_wav_path),
        ],
        text=True,
        timeout=30,
    ).strip()
    dur_sec = round(float(dur_str), 3)
    return {
        "duration_sec": dur_sec,
        "duration_ms": max(1, int(round(dur_sec * 1000))),
        "size_bytes": out_wav_path.stat().st_size,
    }

## Step 7（可选）· 日语原生 vs 中文跨语种 A/B 试听对比

In [ ]:
import time
from pathlib import Path
from IPython.display import Audio, HTML, display

# 勾选 True 后运行本单元格，即可并排对比 3 句日语原生与中文跨语种克隆效果
RUN_AB_COMPARISON = False  #@param {type:"boolean"}

if not RUN_AB_COMPARISON:
    print("已跳过 A/B 对比（如需试听中/日对比，请勾选 RUN_AB_COMPARISON=True 后重跑此单元格）。")
else:
    ab_dir = Path("/content/ab_compare")
    ab_dir.mkdir(parents=True, exist_ok=True)
    for idx, sample in enumerate(AB_SAMPLES, 1):
        display(HTML(f"<h4>[{idx}/{len(AB_SAMPLES)}] {sample['label']}</h4>"))
        for lang_key, field in (("JA 原生（推荐）", "ja_input"), ("ZH 跨语种对照", "zh_input")):
            txt = sample[field]
            t0 = time.monotonic()
            # 首次推理如遇冷 JIT 编译，给予 1800s 宽裕超时
            wav_b = synthesize_drama3_wav(
                text=txt,
                ref_audio_path=REF_WAV_PATH,
                ref_text=REF_TEXT,
                max_new_tokens=320,
                temperature=1.0,
                top_p=0.9,
                timeout_sec=1800.0 if (idx == 1 and field == "ja_input") else 180.0,
            )
            dt = time.monotonic() - t0
            raw_p = ab_dir / f"{sample['id']}_{field}_raw.wav"
            out_p = ab_dir / f"{sample['id']}_{field}.wav"
            raw_p.write_bytes(wav_b)
            pp = postprocess_wav_pcm48k(raw_p, out_p)
            print(f"  {lang_key}: synth={dt:.2f}s | dur={pp['duration_sec']:.2f}s | {txt}")
            display(Audio(filename=str(out_p)))

## Step 8 · 批量合成 16 句花里实乃理 PV 配音（支持多版本抽卡 + 48 kHz 无损后处理）

In [ ]:
import concurrent.futures
import hashlib
import threading
import time
from pathlib import Path

NUM_TAKES = 2  #@param {type:"slider", min:1, max:3, step:1}
TEMPERATURE = 1.0  #@param {type:"number"}
TOP_P = 0.9  #@param {type:"number"}
# 0 = 自动按显卡档位选择并发（A100-80GB=8，A100-40GB=4）
MAX_CONCURRENCY = 0  #@param {type:"integer"}

EFFECTIVE_CONC = int(MAX_CONCURRENCY) if int(MAX_CONCURRENCY) > 0 else int(GPU_PROFILE["default_concurrency"])
RAW_OUT_DIR = Path("/content/pv_raw")
TAKES_OUT_DIR = Path("/content/pv_takes")
RAW_OUT_DIR.mkdir(parents=True, exist_ok=True)
TAKES_OUT_DIR.mkdir(parents=True, exist_ok=True)


def synth_one_take(line: dict, take_idx: int, *, timeout_sec: float = 180.0, show_diag: bool = False) -> dict:
    lid = line["id"]
    text = line["text_tts"]
    spoken_n = count_spoken_chars(text)
    max_tok = int(line.get("max_new_tokens") or min(500, max(260, spoken_n * 8)))
    min_dur = max(0.80, min(2.2, spoken_n * 0.065))

    wav_b = b""
    raw_dur = 0.0
    synth_sec = 0.0
    used_attempt = 0

    for att in range(1, 3):
        used_attempt = att
        diag_stop = threading.Event()
        if show_diag and att == 1:
            def _watch():
                tick = 0
                while not diag_stop.wait(timeout=60.0):
                    tick += 1
                    try:
                        gi = query_gpu_info()
                        sj = sum(1 for _ in Path("/content/sglang_jit_cache").rglob("*") if _.is_file())
                        tc = sum(1 for _ in Path("/content/triton_cache").rglob("*") if _.is_file())
                        print(
                            f"  [warmup +{tick}m] 首次 sm_80 JIT 编译中，请勿中断 | "
                            f"VRAM={gi['vram_used_mib']}/{gi['vram_total_mib']}MiB sglang_jit={sj} triton={tc}",
                            flush=True,
                        )
                    except Exception:
                        pass
            threading.Thread(target=_watch, daemon=True).start()
        try:
            t_syn = time.monotonic()
            wav_b = synthesize_drama3_wav(
                text=text,
                ref_audio_path=REF_WAV_PATH,
                ref_text=REF_TEXT,
                max_new_tokens=max_tok,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                timeout_sec=timeout_sec,
            )
            synth_sec = time.monotonic() - t_syn
        finally:
            diag_stop.set()

        raw_dur = raw_wav_duration_sec(wav_b)
        if raw_dur >= min_dur:
            break
        print(f"  WARN {lid} take#{take_idx} attempt {att} too short ({raw_dur:.2f}s < {min_dur:.2f}s); retrying...", flush=True)

    raw_p = RAW_OUT_DIR / f"{lid}_t{take_idx}.wav"
    out_p = TAKES_OUT_DIR / f"{lid}_t{take_idx}.wav"
    raw_p.write_bytes(wav_b)
    pp = postprocess_wav_pcm48k(raw_p, out_p)
    sha = hashlib.sha256(out_p.read_bytes()).hexdigest()
    return {
        "id": lid,
        "scene_id": line["scene_id"],
        "take": take_idx,
        "wav_path": str(out_p),
        "duration_sec": pp["duration_sec"],
        "duration_ms": pp["duration_ms"],
        "size_bytes": pp["size_bytes"],
        "sha256": sha,
        "raw_dur_sec": round(raw_dur, 3),
        "synth_sec": round(synth_sec, 2),
        "attempts": used_attempt,
    }


jobs = [(ln, t) for ln in LINES for t in range(1, int(NUM_TAKES) + 1)]
print(f"Starting synthesis: {len(LINES)} lines x {NUM_TAKES} takes = {len(jobs)} clips (concurrency={EFFECTIVE_CONC})")

t_all = time.monotonic()
ALL_TAKE_RESULTS: dict[str, list[dict]] = {ln["id"]: [] for ln in LINES}

# 先单独跑第 1 句 Take #1 完成一次性 sm_80 JIT 预热
first_line, first_take = jobs[0]
print(f"[1/{len(jobs)}] Warmup clip: {first_line['id']} take#{first_take} (首次无缓存可能耗时 10~18 分钟)...", flush=True)
r0 = synth_one_take(first_line, first_take, timeout_sec=1800.0, show_diag=True)
ALL_TAKE_RESULTS[r0["id"]].append(r0)
print(
    f"  OK {r0['id']} t{r0['take']}: synth={r0['synth_sec']:.2f}s | dur={r0['duration_sec']:.2f}s ({r0['size_bytes']:,} B)",
    flush=True,
)

# 预热完成后并发跑完剩余全部台词与抽卡版本
rest_jobs = jobs[1:]
done_cnt = 1
t_warm = time.monotonic()
if rest_jobs:
    with concurrent.futures.ThreadPoolExecutor(max_workers=EFFECTIVE_CONC) as ex:
        fut_map = {ex.submit(synth_one_take, ln, t, timeout_sec=240.0, show_diag=False): (ln["id"], t) for ln, t in rest_jobs}
        for fut in concurrent.futures.as_completed(fut_map):
            res = fut.result()
            ALL_TAKE_RESULTS[res["id"]].append(res)
            done_cnt += 1
            print(
                f"[{done_cnt:2d}/{len(jobs)}] OK {res['id']} t{res['take']}: "
                f"synth={res['synth_sec']:.2f}s | dur={res['duration_sec']:.2f}s",
                flush=True,
            )

for lid in ALL_TAKE_RESULTS:
    ALL_TAKE_RESULTS[lid].sort(key=lambda x: x["take"])

# 默认每句自动挑选最接近预计时长的健康 take
SELECTED_TAKES: dict[str, int] = {}
for ln in LINES:
    est = float(ln.get("est_duration_sec") or 4.0)
    cands = ALL_TAKE_RESULTS[ln["id"]]
    best = min(cands, key=lambda r: abs(r["duration_sec"] - est))
    SELECTED_TAKES[ln["id"]] = best["take"]

warm_dt = time.monotonic() - t_warm
print(f"ALL DONE! Warm batch ({len(rest_jobs)} clips) took {warm_dt:.1f}s | Total {time.monotonic() - t_all:.1f}s")

## Step 9 · 逐句试听与抽卡选片（按需修改 `MANUAL_TAKE_OVERRIDES` 后重跑本单元格）

In [ ]:
from IPython.display import Audio, HTML, display

# 如试听后想改选某句的 Take 编号，在此填入 {"line_id": take_number}，例如 {"01_hook_a": 2}
MANUAL_TAKE_OVERRIDES: dict[str, int] = {}

for lid, take_num in MANUAL_TAKE_OVERRIDES.items():
    if lid not in ALL_TAKE_RESULTS:
        raise KeyError(f"Unknown line_id in MANUAL_TAKE_OVERRIDES: {lid}")
    valid_takes = {r["take"] for r in ALL_TAKE_RESULTS[lid]}
    if take_num not in valid_takes:
        raise ValueError(f"Take #{take_num} not available for {lid} (available: {sorted(valid_takes)})")
    SELECTED_TAKES[lid] = int(take_num)

scene_map = {s["scene_id"]: s for s in SCENES}
last_scene = None
for ln in LINES:
    sid = ln["scene_id"]
    if sid != last_scene:
        sc = scene_map[sid]
        display(HTML(f"<hr><h3>Scene {sc['index']} · {sid} — {sc['title_zh']}</h3>"))
        last_scene = sid
    chosen = SELECTED_TAKES[ln["id"]]
    display(
        HTML(
            f"<p><b>{ln['id']}</b> (默认选中 Take #{chosen}, 预计 {ln['est_duration_sec']:.2f}s)<br>"
            f"ZH: {ln['subtitle_zh']}<br>"
            f"JA: {ln['subtitle_ja']}</p>"
        )
    )
    for rec in ALL_TAKE_RESULTS[ln["id"]]:
        star = " ★ SELECTED" if rec["take"] == chosen else ""
        print(f"  Take #{rec['take']}{star}: {rec['duration_sec']:.3f}s ({rec['size_bytes']:,} B)")
        display(Audio(filename=rec["wav_path"]))

## Step 10 · 生成精确 `timings.json` 并打包下载 `mnr_voiceover_pack.zip`

In [ ]:
import datetime
import hashlib
import json
import shutil
import tarfile
import zipfile
from pathlib import Path

SAVE_ZIP_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/moesekai-pv-voiceover"  #@param {type:"string"}
# 勾选后把本次编译出的 sm_80 JIT 缓存存到 Drive (~17 MiB)，下次重跑/重抽卡只需 ~5 分钟！
SAVE_JIT_CACHE_TO_DRIVE = False  #@param {type:"boolean"}

PACK_DIR = Path("/content/mnr_voiceover_pack")
if PACK_DIR.exists():
    shutil.rmtree(PACK_DIR)
(PACK_DIR / "audio").mkdir(parents=True, exist_ok=True)

timing_defaults = PV_SCRIPT.get("timing_defaults", {})
default_gap = float(timing_defaults.get("inter_line_gap_sec", 0.25))
default_tail = float(timing_defaults.get("scene_tail_pad_sec", 0.55))

lines_by_id = {ln["id"]: ln for ln in LINES}
cursor = 0.0
scene_timings = []
line_timings = []

for sc in SCENES:
    sid = sc["scene_id"]
    gap = float(sc.get("inter_line_gap_sec", default_gap))
    pad = float(sc.get("post_scene_pad_sec", default_tail))
    scene_start = round(cursor, 3)
    sc_line_ids = sc["line_ids"]

    for idx_in_scene, lid in enumerate(sc_line_ids):
        ln = lines_by_id[lid]
        chosen_take = SELECTED_TAKES[lid]
        rec = next(r for r in ALL_TAKE_RESULTS[lid] if r["take"] == chosen_take)
        dst_wav = PACK_DIR / "audio" / f"{lid}.wav"
        shutil.copy2(rec["wav_path"], dst_wav)

        line_start = round(cursor, 3)
        dur_sec = float(rec["duration_sec"])
        line_end = round(line_start + dur_sec, 3)
        cursor = line_end

        line_timings.append(
            {
                "id": lid,
                "index": ln["index"],
                "scene_id": sid,
                "speaker": ln["speaker"],
                "selected_take": chosen_take,
                "audio_file": f"audio/{lid}.wav",
                "audio_filename": f"{lid}.wav",
                "start_sec": line_start,
                "end_sec": line_end,
                "duration_sec": round(dur_sec, 3),
                "duration_ms": int(rec["duration_ms"]),
                "size_bytes": int(rec["size_bytes"]),
                "sha256": rec["sha256"],
                "sample_rate": 48000,
                "channels": 1,
                "codec": "pcm_s16le",
                "subtitle_zh": ln["subtitle_zh"],
                "subtitle_ja": ln["subtitle_ja"],
                "text_tts": ln["text_tts"],
            }
        )
        if idx_in_scene < len(sc_line_ids) - 1:
            cursor = round(cursor + gap, 3)

    cursor = round(cursor + pad, 3)
    scene_end = round(cursor, 3)
    scene_timings.append(
        {
            "scene_id": sid,
            "index": sc["index"],
            "title_zh": sc["title_zh"],
            "start_sec": scene_start,
            "end_sec": scene_end,
            "duration_sec": round(scene_end - scene_start, 3),
            "line_ids": list(sc_line_ids),
        }
    )

generated_at = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
TIMINGS_DOC = {
    "schema": "moesekai-pv-timings/1",
    "is_real_mnr_tts": True,
    "source": "colab_drama3_int4",
    "model": MODEL_ID,
    "engine": "sglang-omni",
    "voice_id": PV_SCRIPT["voice"]["voice_id"],
    "voice_display_zh": PV_SCRIPT["voice"]["display_name_zh"],
    "voice_display_ja": PV_SCRIPT["voice"]["display_name_ja"],
    "gpu": GPU_INFO["gpu_name"],
    "gpu_tier": GPU_PROFILE["tier"],
    "generated_at": generated_at,
    "audio_spec": PV_SCRIPT["audio_spec"],
    "total_duration_sec": round(cursor, 3),
    "total_lines": len(line_timings),
    "scenes": scene_timings,
    "lines": line_timings,
}

(PACK_DIR / "timings.json").write_text(json.dumps(TIMINGS_DOC, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(PACK_DIR / "script.json").write_text(json.dumps(PV_SCRIPT, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

manifest_files = []
for p in sorted(PACK_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(PACK_DIR).as_posix()
        manifest_files.append(
            {
                "path": rel,
                "size_bytes": p.stat().st_size,
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            }
        )

PACK_MANIFEST = {
    "schema": "moesekai-pv-voiceover-pack/1",
    "is_real_mnr_tts": True,
    "generated_at": generated_at,
    "total_duration_sec": TIMINGS_DOC["total_duration_sec"],
    "files": manifest_files,
}
(PACK_DIR / "pack_manifest.json").write_text(
    json.dumps(PACK_MANIFEST, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

ZIP_PATH = Path("/content/mnr_voiceover_pack.zip")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(PACK_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(PACK_DIR).as_posix())

zip_sha = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f"Packed {ZIP_PATH}: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MiB | sha256={zip_sha[:16]}...")
print(f"Total PV Voiceover Duration: {TIMINGS_DOC['total_duration_sec']:.2f}s across {len(scene_timings)} scenes / {len(line_timings)} lines")

if SAVE_ZIP_TO_DRIVE or SAVE_JIT_CACHE_TO_DRIVE:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    out_drive = Path(DRIVE_EXPORT_DIR)
    out_drive.mkdir(parents=True, exist_ok=True)
    if SAVE_ZIP_TO_DRIVE:
        dst_zip = out_drive / ZIP_PATH.name
        shutil.copy2(ZIP_PATH, dst_zip)
        print(f"Saved zip backup to Google Drive: {dst_zip}")
    if SAVE_JIT_CACHE_TO_DRIVE:
        jit_tar = out_drive / "sglang_drama3_sm80_jit_cache.tar.gz"
        with tarfile.open(str(jit_tar), "w:gz", compresslevel=6) as tf:
            for prefix, src_d in (("sglang_jit_cache", Path("/content/sglang_jit_cache")), ("triton_cache", Path("/content/triton_cache"))):
                if src_d.exists():
                    for fp in src_d.rglob("*"):
                        if fp.is_file():
                            tf.add(str(fp), arcname=f"{prefix}/{fp.relative_to(src_d).as_posix()}", recursive=False)
        print(f"Saved sm_80 JIT cache tarball to Google Drive: {jit_tar} ({jit_tar.stat().st_size / (1024**2):.2f} MiB)")

try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
    print("Browser download triggered for mnr_voiceover_pack.zip!")
except Exception as exc:
    print(f"Download manually from Colab file browser: {ZIP_PATH} ({exc})")

## Step 11 · 停止推理服务并可选自动释放 Colab GPU 运行时（防空烧 CCU）

In [ ]:
import time

# 确认 mnr_voiceover_pack.zip 已成功下载到本机后，可勾选 True 并运行以立刻释放 A100 停止计费
AUTO_UNASSIGN_RUNTIME = False  #@param {type:"boolean"}

stop_sgl_omni_server()
print("sgl-omni serve stopped cleanly.")

if AUTO_UNASSIGN_RUNTIME:
    print("Waiting 5s before calling google.colab.runtime.unassign() ...")
    time.sleep(5.0)
    from google.colab import runtime as colab_runtime
    colab_runtime.unassign()
else:
    print("运行时仍保持活跃（如需立刻停止计费，请勾选 AUTO_UNASSIGN_RUNTIME=True 重跑，或手动点击「断开连接并删除运行时」）。")